# PE_V2X_Reliability — Project Report (Scheme A)

This report presents the motivation, three-scenario modeling, metric protocol, reproducibility design, and a figure-by-figure discussion of Fig01–Fig07. Figures are shown as PNG previews rendered from the authoritative PDFs.

**Date:** 2026-03-01

## 1 Motivation and problem definition

The key requirement of V2X safety messaging is not “maximum bandwidth” or “minimum average delay”, but to continuously satisfy **reliability and timeliness constraints** required by cooperative perception/decision making under **highly dynamic topology**, **complex propagation environments**, and **shared-channel contention**. In real roads, degraded environments (urban obstruction, tunnels) and busy-hour congestion are not rare corner cases; they are common boundaries that an engineering system must face: urban canyons, interchanges, and long tunnels are widespread, and peak traffic naturally increases channel busy ratio. Therefore, building an **interpretable, reproducible, and extensible** evaluation framework for “V2X reliability and timeliness under degraded environments” has clear engineering significance and research value.

From the application perspective, at least two failure modes must be distinguished, and they affect safety functions differently:

1) **Reliability failure (non-delivery)**: the message does not arrive, decreasing PDR (Packet Delivery Ratio).  
2) **Timeliness failure (late delivery)**: the message arrives after the deadline; for cooperative decision making it is effectively invalid, i.e., a “hidden failure”.

Hence a single metric (e.g., mean delay or a single PDR) cannot support rigorous conclusions: averages hide **tail risk**, and with a deadline, tail inflation directly converts into late arrivals and reduces effective success. To avoid “phenomena without attribution”, this project adopts a system-level modeling approach with explicit mechanisms, and reports under a unified protocol:

- **Reliability decay with distance**: PDR vs distance;  
- **Delay distribution and tail risk**: delay CDF, p95/p99;  
- **Deadline-aware effective success**: `success_phy` (PHY success), `late` (deadline miss), `success` (timely success).

We further constrain degraded environments into three representative scenarios (RefPlus / UrbMaskPlus / TunnelPlus), and introduce a strict two-regime comparison **NoCong / Cong**. All other protocol elements are frozen, and only a congestion toggle introduces contention/collision effects. This supports two system-design questions:

- Does busy-hour congestion become the common dominant bottleneck, compressing environment differences and changing policy choices?  
- With retransmission `ret=0/1/2` as the main enhancement lever, how do gains (higher reliability) and costs (thicker tails, higher late) trade off across scenarios/regimes?

---

## 2 What was built and the overall workflow

The deliverable can be summarized as a **system-level evaluation pipeline**: input generation → event simulation → aggregation → figure production, frozen into a reproducible package by a **run_id output contract + command snapshots**. The purpose is not “one-time results”, but a reusable skeleton that preserves the same metric protocol and evidence chain for future extensions (more degraded environments, RSU, strategy improvements, real-trajectory inputs).

### 2.1 Input generation: making scenarios and traffic explicit and auditable

To avoid “critical settings hidden in code”, inputs are split into traceable files/configs:

- **Mobility input**: trajectories CSV generated from RefPlus geometry + IDM car-following + traffic-signal phases + cross/turn mechanisms.  
- **Building input (UrbMask)**: an urban block represented by rectangles, generating a buildings CSV (footprints, heights, spatial patterns).  
- **Tunnel input (Tunnel)**: tunnel intervals and impairment parameters stored as a tunnel config CSV.

This layer has a clear engineering value: a reviewer does not need the author’s machine paths or every line of code; inputs and command snapshots are sufficient to reproduce the same simulation conditions.

### 2.2 Event simulation: packet-level samples at the granularity of message events

The simulator outputs, per safety message event, three categories:

- **Geometry/state**: distance, link midpoint, scenario states (e.g., NLOS, inside tunnel);  
- **Outcomes**: `success_phy`, `delay_ms`, `late`, `success`;  
- **Evidence fields**: obstruction intensity (`d_min`/`b`), tunnel position (`tunnel_u`), congestion proxies (`n_cs`/CBR/`p_col`/`cong_delay_ms`).

Evidence fields enable figures to show not only “worse”, but also “why worse”.

### 2.3 Aggregation: compressing raw packet logs into reviewer-friendly tables

Packet-level raw logs can be large; the review protocol is aligned to aggregated CSV tables:

- **summary_metrics**: distance-bin PDR, p50/p95/p99, late ratio, and evidence means;  
- **position_heterogeneity**: UrbMask same-distance heterogeneity (within a fixed band, binned by `mid_x`);  
- **tunnel_segments**: Tunnel segmentation table (within a fixed band, binned by `tunnel_u`).

These tables are not “extra files for the sake of it”; they map directly to mechanism explanations: Fig05 requires the heterogeneity table; Fig06 requires segmentation statistics.

### 2.4 Figures: final figures vs transparency drill-down

Two figure tiers are produced:

- **Final figures (Fig01–Fig07)**: unified protocol and high information density; used for reporting/citation.  
- **Deliverable figures (F1/F2/F3)**: full-combination plots for transparency and drill-down; they do not replace final figures.

---

## 3 Significance, value, and contributions

This section follows a paper-style “value–contribution–verifiability” structure, aligned with the actual workload without exaggeration.

### 3.1 Why it matters

**(1) Application-aligned success definition: explicit modeling of “late”**  
In real systems, “received but too late” is equivalent to failure for cooperative decisions. By separating `success_phy` and `success`, and logging `late`, the conclusions are directly relevant to safety constraints rather than purely networking averages.

**(2) Mechanization of degraded-environment phenomena: heterogeneity and segmentation are quantifiable**  
Urban canyon difficulty is not only a lower mean, but same-distance heterogeneity; tunnel difficulty is segmented impairment and thicker tails. Dedicated tables and evidence fields turn these into auditable statistics.

**(3) Explainable trade-offs under congestion: avoiding single-metric bias**  
Retransmissions often gain reliability but thicken tails; under congestion, PHY gains may be offset by more late. Fig03 decomposition and Fig04 proxy evidence turn “trade-offs” into measurable, causal components.

**(4) Strong reproducibility: reviewers can validate on any machine**  
The combination of run_id contract, command snapshots, tables, and final figures enables verification without relying on the author’s folder structure, while preserving a stable skeleton for extension.

### 3.2 Concrete contributions

**Contribution 1: Standardized scenario construction with interpretable modeling**
- RefPlus: controlled baseline (geometry + traffic + signals) for attribution;  
- UrbMaskPlus: continuous obstruction `d_min→b` + LOS/NLOS mixture + reflection-equivalent term for same-distance heterogeneity;  
- TunnelPlus: `tunnel_u` segmentation + bell+fade impairment + tail-delay injection for segmented degradation and tail risk.

**Contribution 2: Congestion proxy chain and Cong-only ablation**
- Explicit path “traffic density → channel busy → collisions/queueing → thicker tails” through `n_cs`/CBR/`p_col`/`cong_delay_ms`;  
- NoCong vs Cong differs only by a toggle, enabling clean attribution.

**Contribution 3: Unified gain–cost evaluation for ret=0/1/2**
- Not only PDR gain, but also p95/p99 and late risk are reported;  
- Strategy choice becomes a timeliness-constrained trade-off rather than “maximize PDR”.

**Contribution 4: Auditable evidence chain (figure–table–field loop)**
- Fig05 ⇔ position_heterogeneity ⇔ (`d_min`, `b`, `g_refl`);  
- Fig06 ⇔ tunnel_segments ⇔ (`tunnel_u`, tail delay);  
- Fig03/Fig04 ⇔ (`success_phy`, `late`, `p_col`, `cong_delay_ms`).

### 3.3 How to validate quickly

A minimal review path:

1) Read Fig01–Fig07;  
2) Check run_commands (frozen protocol; minimal NoCong/Cong difference);  
3) Verify table fields and binning;  
4) Drill down with deliverable figures if desired.

---

## 4 Scenario construction (detailed, with equations)

This chapter follows a paper-style “system model / scenario setting” description, using a unified coordinate and time discretization, then detailing RefPlus / UrbMaskPlus / TunnelPlus. Note that this is a system-level evaluation framework aiming at explainability under controlled complexity (not full ray tracing nor full-stack protocol reproduction).

### 4.0 Unified conventions: coordinates, discrete time, and link midpoint

- Planar coordinates: corridor along \(x\), lateral offset \(y\).  
- Discrete time step \(\Delta t\); messages generated at frequency \(f_m\).  
- For a Tx/Rx link:

$$
d=\|\mathbf{p}_{tx}-\mathbf{p}_{rx}\|_2,\qquad 
\mathbf{p}_{mid}=\frac{\mathbf{p}_{tx}+\mathbf{p}_{rx}}{2},\qquad
x_{mid}=\mathbf{p}_{mid,x}.
$$

### 4.1 RefPlus: controlled baseline (geometry + IDM + signals + cross/turn)

#### 4.1.1 Geometry: corridor centerline and multiple lanes

An illustrative smooth S-curve:

$$
y_c(x)=A\sin\!\Big(2\pi\,\frac{x-x_0}{x_1-x_0}\Big)\cdot \mathbb{I}_{[x_0,x_1]}(x).
$$

Lane centerlines:

$$
y_{\ell}^{(\pm)}(x)=y_c(x)\ \pm\ \Big(\frac{g}{2}+\big(\ell+\tfrac{1}{2}\big)w\Big),
\qquad \ell\in\{0,1,2\}.
$$

#### 4.1.2 Traffic: IDM car-following (queues, platoons, stop-and-go waves)

$$
\dot v = a\Big[1-\Big(\frac{v}{v_0}\Big)^{\delta}-\Big(\frac{s^{\ast}(v,\Delta v)}{s}\Big)^2\Big],
\qquad
s^{\ast}(v,\Delta v)=s_0+vT+\frac{v\Delta v}{2\sqrt{ab}}.
$$

#### 4.1.3 Signals and intersections: phase and offset

$$
\phi_{main}(t)=\mathbb{I}\big(\ (t+\tau)\bmod C\in[0,G)\ \big).
$$

### 4.2 UrbMaskPlus: urban canyon obstruction (buildings + continuous obstruction + reflection-equivalent)

Buildings are rectangles \(\mathcal{B}=\{B_k\}\). Minimum distance:

$$
d_{min}=\min_{B_k\in\mathcal{B}} \ \mathrm{dist}\big(\overline{\mathbf{p}_{tx}\mathbf{p}_{rx}},\partial B_k\big).
$$

Obstruction intensity:

$$
b=\exp\!\left(-\Big(\frac{d_{min}}{T}\Big)^2\right).
$$

LOS/NLOS mixture:

$$
p_{succ}(d,b)=(1-b)\,p_{LOS}(d)+b\,p_{NLOS}(d).
$$

Optional reflection-equivalent term:

$$
G_{refl}(d_{min},b)=G_{max}\exp(-d_{min}/d_0)\,b.
$$

Same-distance heterogeneity bins:

$$
\mathcal{S}_j=\{\,\text{links}: d\in[d_1,d_2],\ x_{mid}\in I_j\,\}.
$$

### 4.3 TunnelPlus: segmented degradation (tunnel_u + bell+fade + tail delay)

Normalized tunnel position:

$$
u=\mathrm{clip}\Big(\frac{x_{mid}-x_0}{x_1-x_0},0,1\Big).
$$

Bell shape:

$$
\mathrm{bell}(u)=\sin^2(\pi u),\qquad \mathrm{bell}_\gamma(u)=\mathrm{bell}(u)^\gamma.
$$

Tail delay injection (illustrative):

$$
D = D_0 + b_{tun}\cdot D_{extra},\qquad D_{extra}\sim \mathrm{Exp}(\lambda).
$$

Segment bins:

$$
\mathcal{T}_j=\{\,\text{links}: d\in[d_1,d_2],\ u\in J_j\,\}.
$$

### 4.4 Evidence-field design (for a closed loop of phenomenon–mechanism–implication)

- UrbMask: `d_min`, `b`, `g_refl` ⇔ heterogeneity (Fig05)  
- Tunnel: `tunnel_u` ⇔ segmentation (Fig06)  
- Cong: `n_cs`, CBR, `p_col`, `cong_delay_ms` ⇔ composite-failure decomposition (Fig03/04)

---

## 5 Modeling and metrics

Define \(S_{phy}\in\{0,1\}\), \(L\in\{0,1\}\), and \(S=S_{phy}(1-L)\):

$$
\mathrm{PDR}\approx \mathrm{PDR}_{phy}\cdot (1-\mathrm{LateRatio}_{phy}).
$$

Conceptual retransmission and delay composition:

$$
\mathbb{P}(S_{phy}=1)=1-\prod_{k=0}^{ret}(1-p_k),
\qquad
D = D_{base} + D_{prop} + D_{queue} + D_{cong} + D_{retry}.
$$

Congestion proxies (illustrative):

$$
\mathrm{CBR}=\min\big(1,\ (n_{cs}-1)\lambda\tau_{air}\big),
\qquad
p_{col}=1-\exp(-k\,\mathrm{CBR}),
\qquad
D_{cong}\sim \mathrm{Exp}(\beta(\mathrm{CBR})).
$$

---

## 6 Reproducibility and audit (formal statement)

Let `$ROOT` denote the repository root. Outputs follow the `runs/<run_id>/` contract and include `run_commands.txt`. The difference between NoCong and Cong is restricted to a congestion toggle for attribution. Recommended reproduction tiers: smoke → small → S20.

# Figure-by-figure decomposition and discussion (Raw-derived, S20)

## 1. Data source and statistical workflow (raw → cache → figures)

This result set is based on two strictly controlled final experiments: **A_Final_NoCong_S20** and **A_Final_Cong_S20**. Their shared configuration (given by the final command logs and project configuration files) is identical: scenarios **Ref / UrbMask / Tunnel**, retransmission strategies **ret=0/1/2**, random seeds **seed=1–20**, and business-level statistics focusing on **0–200 m**. The only difference is whether the congestion/competition mechanism is enabled, forming a strict ablation (NoCong vs Cong).

To avoid concerns that “tables are too sparse so curves are not smooth/credible”, the final results are not drawn directly from tables. Instead, a two-stage statistics pipeline is used:
(1) aggregate raw data by distance bins into counts and histogram statistics required for quantiles;
(2) cache the aggregated results into `.mat` (for reproducibility and fast re-plotting), then use MATLAB to produce the 7 final figures.
Cache filenames encode key resolutions, e.g., **distance bin=2 m**, **delay histogram bin=0.25 ms**, and the mechanism-analysis band **band=80–100 m** (for UrbMask heterogeneity and Tunnel segmentation).

---

## 2. Metric definitions and layers of “success” (avoid ambiguity)

With an application deadline, “success” has at least two layers: **PHY success** and **timely success**. This report uses three complementary metrics:

1. **timely PDR (PDR or timely_rate in figures)**
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\frac{N*{\mathrm{success}}(d)}{N_{\mathrm{total}}(d)}
$$
where \(N_{\mathrm{success}}\) counts packets arriving within the deadline.

2. **PHY success rate (phy_rate in figures)**
$$
\mathrm{PDR}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{success_phy}}(d)}{N_{\mathrm{total}}(d)}
$$
where \(N_{\mathrm{success_phy}}\) counts PHY decode successes (timeliness not required).

3. **late ratio among PHY successes (late_ratio_phy in figures)**
$$
\mathrm{late_ratio}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{late}}(d)}{N_{\mathrm{success_phy}}(d)}
$$
where \(N_{\mathrm{late}}\) counts “PHY success but beyond the deadline”.

They satisfy the key identity (explicit in Fig03):
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\mathrm{PDR}*{\mathrm{phy}}(d)\cdot\left(1-\mathrm{late_ratio}_{\mathrm{phy}}(d)\right)
$$

---

# Fig01–Fig07 decomposition (as main text + captions)

## Fig01: PDR vs distance (dist≤200 m; three-scenario comparison; NoCong solid, Cong dotted)

![](assets/final_figures_A_preview/Fig01_PDR_Focus.png)

**Meaning and role**  
Fig01 is the “main phenomenon” figure: under degraded environments and congestion, how does reliability (timely PDR) degrade with distance, and how much benefit does ret bring?

**Observation 1: NoCong — propagation-dominated degradation and monotonic ret gain**  
Fig07 anchors over dist≤200 m:

* Ref: PDR = 0.439 (ret0) → 0.537 (ret1) → 0.582 (ret2)
* UrbMask: PDR = 0.462 (ret0) → 0.584 (ret1) → 0.644 (ret2)
* Tunnel: PDR = 0.368 (ret0) → 0.489 (ret1) → 0.542 (ret2)

**Observation 2: Cong — global collapse and scenario differences become second-order**  
Fig07 anchors:

* Ref: PDR = 0.054 (ret0) → 0.090 (ret1) → 0.117 (ret2)
* UrbMask: PDR = 0.057 (ret0) → 0.097 (ret1) → 0.126 (ret2)
* Tunnel: PDR = 0.049 (ret0) → 0.082 (ret1) → 0.107 (ret2)

**Report-ready sentence**  
NoCong: ret yields stable reliability gains with clear scenario differences; Cong: timely PDR collapses and scenario gaps converge, indicating congestion dominates.

---

## Fig02: Tail delays (p95/p99) vs distance (NoCong/Cong; ret0 vs ret2)

![](assets/final_figures_A_preview/Fig02_TailDelay_p95p99_Ret0Ret2.png)

**NoCong**: tails rise with ret but remain far below the deadline (late≈0). Fig07 p95 anchors:

* Ref: p95=2.9 ms (ret0) → 22.4 ms (ret2)
* UrbMask: p95=3.9 ms (ret0) → 22.9 ms (ret2)
* Tunnel: p95=3.9 ms (ret0) → 22.9 ms (ret2)

**Cong**: tails are raised into the 70–100 ms range and the deadline becomes active. Fig07 p95 anchors:

* Ref: p95=69.6 ms (ret0) → 79.9 ms (ret2)
* UrbMask: p95=70.1 ms (ret0) → 79.9 ms (ret2)
* Tunnel: p95=69.1 ms (ret0) → 79.6 ms (ret2)

---

## Fig03: Cong-only decomposition (PHY success, timely success, late penalty)

![](assets/final_figures_A_preview/Fig03_Cong_Decomposition_3Curves.png)

Reading: blue = \(\mathrm{PDR}_{phy}\), green = \(\mathrm{PDR}_{timely}\), orange (right axis) = \(\mathrm{late\_ratio}_{phy}\). Identity:
$$
\mathrm{PDR}*{\mathrm{timely}}=\mathrm{PDR}*{\mathrm{phy}}\cdot(1-\mathrm{late_ratio}_{\mathrm{phy}})
$$

Fig07 late% quantifies the double-edged effect of ret under Cong:

* Ref: late=5.5% (ret0) → 8.7% (ret2)
* UrbMask: late=5.4% (ret0) → 8.4% (ret2)
* Tunnel: late=5.1% (ret0) → 7.9% (ret2)

---

## Fig04: Congestion proxy evidence (mean ± std band)

![](assets/final_figures_A_preview/Fig04_CongProxy_MeanStdBand.png)

Across dist≤200 m, proxies (CBR, \(p_{col}\), congestion delay) show similar means and narrow variance bands across scenarios, supporting “congestion dominates and compresses scenario differences”.

---

## Fig05: UrbMask spatial heterogeneity (heterogeneity and congestion amplification)

![](assets/final_figures_A_preview/Fig05_UrbMask_Heterogeneity_LinesAndRatio.png)

NoCong: within band 80–100 m, PDR_band varies structurally along mid_x.  
Ratio Cong/NoCong: congestion penalizes weak regions more strongly (lower ratio), indicating location-dependent risk concentration.

---

## Fig06: Tunnel segmented effect (inside vs outside)

![](assets/final_figures_A_preview/Fig06_Tunnel_InsideOutside_Bars_UnifiedY.png)

NoCong: inside is significantly weaker than outside; ret improves both but segmentation remains.  
Cong: inside approaches unusable and late penalties appear; consistent with Fig02/03.

---

## Fig07: Summary matrices (dist≤200 m anchors)

![](assets/final_figures_A_preview/Fig07_Summary_Matrices.png)

NoCong: strong PDR gains with tail cost; late≈0.  
Cong: low absolute PDR dominated by contention; p95 near deadline; late% rises with ret.

---

# Cross-figure synthesis (phenomenon–mechanism–implication)

1) Bottleneck shift: propagation → congestion (Fig01+Fig04).  
2) Gain–cost boundary: ret gains vs tail/late costs (Fig02/03/07).  
3) Structured degradation: UrbMask heterogeneity amplified; Tunnel segmentation extremized (Fig05/06).  
4) Implication: retransmission alone cannot optimize both reliability and timeliness; joint design with congestion control is required.

---

# Recommended limitation statements

1) Resolution/smoothing: visualization only; numeric anchors via Fig06/Fig07.  
2) Quantiles: histogram-based; mask bins with insufficient samples.  
3) Proxies: controllable internal indicators for attribution; not a full-detail standard stack reproduction.

## Code structure overview

Project code is under `03_sim_A/py/` and `03_sim_A/py/modules/`. The end-to-end pipeline is orchestrated by `run_pipeline_A.py`. Detailed per-file documentation is provided in Notebook 2.